<a href="https://colab.research.google.com/github/francji1/01RAD/blob/main/code/01RAD_Ex02_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01RAD - Homework 02: Regression to the mean (Galton's heights)

In 1886 Francis Galton compared the heights of 1,078 fathers and their adult sons and noticed that sons of tall fathers tend to be shorter than their fathers, and sons of short fathers taller. He called the phenomenon *regression towards mediocrity*; the name stuck to the whole method (Lecture 1, history). The data are the `fsdata.csv` file from Exercise 01 (heights in inches).

Model of Lecture 1: $Y_i = \beta_0 + \beta_1 x_i + e_i$ with $x_i$ the father's height and $Y_i$ the son's height, independent errors $e_i \sim (0, \sigma^2)$. Tools of Lecture 2 (slides pp. 14–32 of the lecture PDF): standard errors $\widehat{\sigma}(\widehat{\beta}_j) = s_n \sqrt{\delta_j}$, **confidence intervals** (Czech: intervaly spolehlivosti) $\widehat{\beta}_j \pm t_{1-\alpha/2}(n-2)\,\widehat{\sigma}(\widehat{\beta}_j)$ and the **tests** (testy hypotéz) with statistics $T_j = (\widehat{\beta}_j - b)/\widehat{\sigma}(\widehat{\beta}_j)$ compared with $t_{1-\alpha/2}(n-2)$.

## Submission
Work through the tasks in this notebook (code plus short written answers in markdown cells; Python or R, as you prefer). Submit through GitHub **before the next exercise**: fork `francji1/01RAD`, save the notebook into the `HW/` folder as `01RAD_Ex02_HW_<Surname>.ipynb`, with your surname in place of `<Surname>` and without the brackets, for example `01RAD_Ex02_HW_Novak.ipynb` (team: surnames joined by an underscore), and open a pull request to `main`. The pull request should add your one file only; do not modify or delete other files. A selected student solution, and a reference solution if needed, will be published in `HW/` a week later.

Useful calls: `stats.t.ppf(1 - alpha/2, df)` for the $t$ quantile, `stats.t.sf(abs(T), df)` and `stats.t.cdf(T, df)` for tail probabilities, `model.conf_int(alpha)`, `model.t_test('father = 1')`, `model.predict(pd.DataFrame({'father': [...]}))`, `np.corrcoef`, `df.sample(n, random_state=...)`.

## Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid', palette='deep')

url_fsdata = 'https://raw.githubusercontent.com/francji1/01RAD/main/data/fsdata.csv'
fsdata = pd.read_csv(url_fsdata, index_col=0)
print(f'Rows: {fsdata.shape[0]}, columns: {fsdata.shape[1]}')
fsdata.describe().T

Rows: 1078, columns: 2


,count,mean,std,min,25%,50%,75%,max
father,1078.0,67.687097,2.744868,59.00800,65.787735,67.76660,69.60298,75.43393
son,1078.0,68.684070,2.814702,58.50708,66.931232,68.61582,70.46597,78.36479


## Task 01 - Fit and recap
Fit the regression of `son` on `father`. Compute $\widehat{\beta}_0$, $\widehat{\beta}_1$, $s_n$ and both standard errors by hand (formulas of Lectures 1 and 2) and check them against `statsmodels` (R: `summary(lm(son ~ father))`). Interpret $\widehat{\beta}_1$ in one sentence.

In [2]:
# your solution
model = smf.ols('son ~ father', data=fsdata).fit()
print(model.summary().tables[1])

x = fsdata['father'].to_numpy(dtype=float)
y = fsdata['son'].to_numpy(dtype=float)
n = len(x)
x_bar, y_bar = x.mean(), y.mean()
S_xx = np.sum((x - x_bar) ** 2)

beta1_hat = np.sum((x - x_bar) * y) / S_xx
beta0_hat = y_bar - beta1_hat * x_bar
resid = y - beta0_hat - beta1_hat * x
SSE = np.sum(resid ** 2)
s_n2 = SSE / (n - 2)
s_n = np.sqrt(s_n2)

delta_0 = 1 / n + x_bar ** 2 / S_xx
delta_1 = 1 / S_xx
se_beta0 = s_n * np.sqrt(delta_0)
se_beta1 = s_n * np.sqrt(delta_1)

pd.DataFrame({
    'quantity': ['beta0_hat', 'beta1_hat', 's_n', 'SE(beta0_hat)', 'SE(beta1_hat)'],
    'manual': [beta0_hat, beta1_hat, s_n, se_beta0, se_beta1],
    'statsmodels': [model.params['Intercept'], model.params['father'], np.sqrt(model.scale),
                    model.bse['Intercept'], model.bse['father']],
})

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     33.8866      1.832     18.493      0.000      30.291      37.482
father         0.5141      0.027     19.006      0.000       0.461       0.567


,quantity,manual,statsmodels
0,beta0_hat,33.886604,33.886604
1,beta1_hat,0.514093,0.514093
2,s_n,2.436556,2.436556
3,SE(beta0_hat),1.832354,1.832354
4,SE(beta1_hat),0.027049,0.027049


If the father is 1 inch taller, the mean height of his sons is estimated to be 0.514 inches higher.

## Task 02 - Confidence intervals
Compute the 95 % and the 99 % confidence intervals for $\beta_0$ and $\beta_1$ by hand, using the $t$ quantile, and compare with `conf_int` (R: `confint`). Why is the 99 % interval wider? Compare the widths of the intervals for $\beta_0$ and $\beta_1$ in absolute terms and relative to the estimates: which parameter is estimated more precisely? Relate your answer to the remark in Lecture 2.

In [3]:
# your solution
for alpha in (0.05, 0.01):
    t_q = stats.t.ppf(1 - alpha / 2, df=n - 2)
    ci_manual = pd.DataFrame({
        'lower': [beta0_hat - t_q * se_beta0, beta1_hat - t_q * se_beta1],
        'upper': [beta0_hat + t_q * se_beta0, beta1_hat + t_q * se_beta1],
    }, index=['Intercept', 'father'])
    ci_manual['width'] = ci_manual['upper'] - ci_manual['lower']
    ci_manual['width / estimate'] = ci_manual['width'] / np.array([beta0_hat, beta1_hat])
    print(f"{100 * (1 - alpha):.0f} % CI, t quantile = {t_q:.4f}")
    print("manual:")
    print(ci_manual)
    print("conf_int (R: confint(mod)):")
    print(model.conf_int(alpha=alpha), "\n")

95 % CI, t quantile = 1.9622
manual:
               lower      upper     width  width / estimate
Intercept  30.291213  37.481996  7.190784          0.212201
father      0.461019   0.567167  0.106148          0.206477
conf_int (R: confint(mod)):
                   0          1
Intercept  30.291213  37.481996
father      0.461019   0.567167 

99 % CI, t quantile = 2.5804
manual:
               lower      upper     width  width / estimate
Intercept  29.158387  38.614822  9.456434          0.279061
father      0.444296   0.583890  0.139593          0.271533
conf_int (R: confint(mod)):
                   0          1
Intercept  29.158387  38.614822
father      0.444296   0.583890 



The intervals by hand agree with conf_int. The 99 % interval is wider because it uses the larger quantile $t_{0.995}(1076) = 2.580$ instead of $t_{0.975}(1076) = 1.962$. In absolute terms the interval for $\beta_0$ (width 7.19) is much wider than for $\beta_1$ (width 0.106), so the slope is estimated more precisely. Relative to the estimates both widths are about 21 %.

## Task 03 - Is the father's height informative?
Test $H_0: \beta_1 = 0$ against $H_1: \beta_1 \neq 0$ at $\alpha = 0.05$: report $T_1$, the critical value $t_{0.975}(n-2)$, the $p$-value and the decision. What does the rejection say about the relationship, and what does it *not* say (think of the lecture's warning about linearity)?

In [4]:
# your solution
alpha = 0.05
T1 = beta1_hat / se_beta1
t_crit = stats.t.ppf(1 - alpha / 2, df=n - 2)
p_value = 2 * stats.t.sf(abs(T1), df=n - 2)

print(f'T1 = {T1:.3f}')
print(f't_0.975(n-2) = {t_crit:.4f}')
print(f'p-value = {p_value:.3e}')
print('reject H0' if abs(T1) > t_crit else 'do not reject H0')
print(model.summary().tables[1])

T1 = 19.006
t_0.975(n-2) = 1.9622
p-value = 1.121e-69
reject H0
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     33.8866      1.832     18.493      0.000      30.291      37.482
father         0.5141      0.027     19.006      0.000       0.461       0.567


$|T_1| = 19.01 > t_{0.975}(1076) = 1.962$ and $p \approx 1.1 \cdot 10^{-69}$, so we reject $H_0: \beta_1 = 0$. The son's height has a linear trend in the father's height. The test does not say that the relationship is linear.

## Task 04 - Regression to the mean
If sons were on average exactly as tall as their fathers plus a constant, the slope would be $\beta_1 = 1$. Test $H_0: \beta_1 = 1$ against $H_1: \beta_1 \neq 1$ (the test with a given candidate value $b_1$ from Lecture 2) and also against the **one-sided alternative** (jednostranná alternativa) $H_1: \beta_1 < 1$: the lecture treats two-sided alternatives, for the one-sided one use the same statistic $T = (\widehat{\beta}_1 - 1)/\widehat{\sigma}(\widehat{\beta}_1)$, reject when $T < -t_{1-\alpha}(n-2)$, and the $p$-value is $P(t(n-2) < T)$. Check the duality of the two-sided test with the 95 % confidence interval from Task 02. Explain in your own words why Galton spoke of *regression* (regrese k průměru) and what the estimated slope says about sons of very tall and very short fathers.

In [5]:
# your solution
alpha = 0.05
T = (beta1_hat - 1) / se_beta1

# two-sided alternative H1: beta1 != 1
t_crit_two = stats.t.ppf(1 - alpha / 2, df=n - 2)
p_two = 2 * stats.t.sf(abs(T), df=n - 2)

# one-sided alternative H1: beta1 < 1
t_crit_one = stats.t.ppf(1 - alpha, df=n - 2)
p_one = stats.t.cdf(T, df=n - 2)

print(f'T = {T:.3f}')
print(f'two-sided: critical value {t_crit_two:.4f}, p-value {p_two:.3e}')
print(f'one-sided: reject if T < {-t_crit_one:.4f}, p-value {p_one:.3e}')
print(model.t_test('father = 1'))
print('95 % CI for beta1:', model.conf_int(0.05).loc['father'].values)

T = -17.964
two-sided: critical value 1.9622, p-value 2.610e-63
one-sided: reject if T < -1.6463, p-value 1.305e-63
                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.5141      0.027    -17.964      0.000       0.461       0.567
95 % CI for beta1: [0.46101879 0.56716729]


$T = -17.96$. Two-sided test: $|T| > t_{0.975}(1076) = 1.962$, $p \approx 2.6 \cdot 10^{-63}$, we reject $H_0: \beta_1 = 1$. One-sided test: $T < -t_{0.95}(1076) = -1.646$, $p \approx 1.3 \cdot 10^{-63}$, we reject $H_0$ in favour of $H_1: \beta_1 < 1$. The value 1 lies outside the 95 % confidence interval $(0.461, 0.567)$, which agrees with the two-sided test.

Since $\widehat{\beta}_1 = 0.514 < 1$, one inch of the father's height adds on average only about half an inch to the son's height. Sons of very tall fathers are on average shorter than their fathers, sons of very short fathers taller. Their heights move back (regress) towards the mean.

## Task 05 - The intercept
Test $H_0: \beta_0 = 0$. Then write $\widehat{\beta}_0 = \overline{y} - \widehat{\beta}_1 \overline{x}$, plug in the sample means and explain why the intercept is far from zero even though sons are on average only about one inch taller than their fathers. Would a model through the origin make sense here?

In [6]:
# your solution
T0 = beta0_hat / se_beta0
p0 = 2 * stats.t.sf(abs(T0), df=n - 2)
print(f"T0 = {T0:.3f}, critical value {stats.t.ppf(0.975, df=n - 2):.4f}, p-value {p0:.3e}")

print(f"x_bar = {x_bar:.3f}")
print(f"y_bar = {y_bar:.3f}")
print(f"beta1_hat * x_bar = {beta1_hat * x_bar:.3f}")
print(f"y_bar - beta1_hat * x_bar = {y_bar - beta1_hat * x_bar:.3f}")
print(f"y_bar - x_bar = {y_bar - x_bar:.3f}")

T0 = 18.493, critical value 1.9622, p-value 1.604e-66
x_bar = 67.687
y_bar = 68.684
beta1_hat * x_bar = 34.797
y_bar - beta1_hat * x_bar = 33.887
y_bar - x_bar = 0.997


$|T_0| = 18.49 > t_{0.975}(1076) = 1.962$, $p \approx 1.6 \cdot 10^{-66}$, so we reject $H_0: \beta_0 = 0$.

$\widehat{\beta}_0 = \overline{y} - \widehat{\beta}_1 \overline{x} = 68.684 - 0.514 \cdot 67.687 = 33.887$. The intercept would be close to $\overline{y} - \overline{x} = 0.997$ only if the slope were 1. With slope 0.514 the line must start much higher.

A model through the origin does not make sense: a father of height 0 lies far outside the observed heights (59 to 75 inches), and the test rejects $\beta_0 = 0$.

## Task 06 - Point predictions
Predict the mean height of sons whose fathers are 60, 64, 68, 72 and 76 inches tall and tabulate the difference *predicted son minus father*. For which father's height do the two coincide? (Confidence and prediction intervals for these predictions come in Lecture 4; here point predictions are enough.)

In [7]:
# your solution
fathers = np.array([60, 64, 68, 72, 76])
pred = model.predict(pd.DataFrame({'father': fathers}))
pred_table = pd.DataFrame({'father': fathers,
                           'predicted son': pred.values,
                           'predicted son - father': pred.values - fathers})
print(pred_table)

x_star = beta0_hat / (1 - beta1_hat)
print(f'predicted son = father for father = {x_star:.2f} inches')

   father  predicted son  predicted son - father
0      60      64.732187                4.732187
1      64      66.788559                2.788559
2      68      68.844931                0.844931
3      72      70.901303               -1.098697
4      76      72.957675               -3.042325
predicted son = father for father = 69.74 inches


Sons of short fathers are predicted taller than their fathers (+4.73 inches for 60 inches), sons of tall fathers shorter (−3.04 inches for 76 inches). The two coincide when $\widehat{\beta}_0 + \widehat{\beta}_1 x = x$, i.e. for $x = \widehat{\beta}_0 / (1 - \widehat{\beta}_1) = 69.74$ inches.

## Task 07 - Reverse regression
Fit the reverse regression of `father` on `son` and report its slope. Verify numerically that the product of the two slopes equals the square of the sample correlation coefficient $r$ of the two heights (`np.corrcoef`), and explain why both slopes are smaller than 1. Is regression to the mean a causal effect of fathers on sons, or a statistical property of the two distributions?

In [8]:
# your solution
model_rev = smf.ols('father ~ son', data=fsdata).fit()
beta1_rev = model_rev.params['son']
r = np.corrcoef(x, y)[0, 1]

print(f'slope son ~ father: {beta1_hat:.4f}')
print(f'slope father ~ son: {beta1_rev:.4f}')
print(f'product of slopes:  {beta1_hat * beta1_rev:.4f}')
print(f'r = {r:.4f}, r^2 = {r ** 2:.4f}')
print(f'sd father = {x.std(ddof=1):.3f}, sd son = {y.std(ddof=1):.3f}')

slope son ~ father: 0.5141
slope father ~ son: 0.4889
product of slopes:  0.2513
r = 0.5013, r^2 = 0.2513
sd father = 2.745, sd son = 2.815


The slope of the reverse regression is 0.489, and $0.514 \cdot 0.489 = 0.2513 = r^2$, because the two slopes are $r\, s_y/s_x$ and $r\, s_x/s_y$. Both are smaller than 1 because $r = 0.50$ and $s_x \approx s_y$ (2.745 and 2.815 inches).

Regression to the mean is not a causal effect of fathers on sons, since it works in both directions. It is a statistical property of two variables with similar spread and correlation smaller than 1.

## Task 08 - Sample size and precision
Draw random subsamples of 30, 100 and 300 father-son pairs (fix the random seed), fit the model in each and compare the widths of the 95 % intervals for $\beta_1$ with the full-sample interval. Explain the pattern through $\mathrm{Var}(\widehat{\beta}_1) = \sigma^2 / S_{xx}$. *Optional:* repeat the subsampling with $n = 100$ five hundred times, treat the full-sample $\widehat{\beta}_1$ as the true value and estimate how often the 95 % interval covers it.

In [9]:
# your solution
ci_full = model.conf_int(0.05).loc['father']
width_full = ci_full[1] - ci_full[0]

rows = []
for size in (30, 100, 300):
    sub = fsdata.sample(size, random_state=2026)
    m_sub = smf.ols('son ~ father', data=sub).fit()
    ci = m_sub.conf_int(0.05).loc['father']
    S_xx_sub = np.sum((sub['father'] - sub['father'].mean()) ** 2)
    rows.append([size, m_sub.params['father'], ci[0], ci[1], ci[1] - ci[0],
                 (ci[1] - ci[0]) / width_full, S_xx_sub])
rows.append([n, beta1_hat, ci_full[0], ci_full[1], width_full, 1.0, S_xx])

print(pd.DataFrame(rows, columns=['n', 'beta1_hat', 'lower', 'upper', 'width',
                                  'width / full width', 'S_xx']))

# optional: coverage with n = 100, full-sample estimate as the true value
n_rep = 500
covered = 0
for i in range(n_rep):
    sub = fsdata.sample(100, random_state=i)
    ci = smf.ols('son ~ father', data=sub).fit().conf_int(0.05).loc['father']
    covered += (ci[0] <= beta1_hat <= ci[1])
print(f'coverage of the 95 % interval: {covered / n_rep:.3f}')

      n  beta1_hat     lower     upper     width  width / full width  \
0    30   0.248032 -0.058798  0.554863  0.613661            5.781151   
1   100   0.371847  0.193584  0.550109  0.356525            3.358738   
2   300   0.537262  0.431871  0.642654  0.210783            1.985732   
3  1078   0.514093  0.461019  0.567167  0.106148            1.000000   

          S_xx  
0   110.933003  
1   640.433119  
2  1917.101200  
3  8114.443905  
coverage of the 95 % interval: 0.970


The smaller the subsample, the wider the 95 % interval for $\beta_1$. It is 5.8, 3.4 and 2.0 times the full-sample width for n = 30, 100 and 300. The reason is $\mathrm{Var}(\widehat{\beta}_1) = \sigma^2 / S_{xx}$, where $S_{xx}$ grows roughly proportionally to $n$ (111, 640, 1917 and 8114 here), so the width decreases roughly like $1/\sqrt{n}$.

Optional part: in 500 subsamples of size 100 the 95 % interval covered the full-sample value $\widehat{\beta}_1 = 0.514$ in 97.0 % of cases, close to the nominal 95 %.